# LeetCode 1826 — Faulty Sensor bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

Link đề: https://leetcode.com/problems/faulty-sensor/

Độ khó: **Easy** · Chủ đề: Array, Two Pointers

```python
class Solution:
    def badSensor(self, sensor1: List[int], sensor2: List[int]) -> int:

```

**Mục tiêu học:**

1. Đọc hiểu một đề "mô phỏng lỗi dữ liệu": **mất một phần tử → phần sau dịch trái → cuối mảng bị thay bằng giá trị ngẫu nhiên**.
2. Viết **brute force mô phỏng** đúng từng chữ của đề (thử mọi vị trí bị mất).
3. Rút ra insight: mọi thứ quyết định tại **vị trí khác nhau đầu tiên** `i`; sau đó chỉ cần so sánh hai đoạn **lệch nhau một vị trí**.
4. Hiểu vì sao phải trả `-1` khi **cả hai** khả năng đều hợp lệ (không xác định được).
5. Dùng slicing so sánh đoạn và kỹ thuật hai con trỏ không cần tạo bản sao.

## 1. Bài toán yêu cầu gì?

Một thí nghiệm dùng **hai cảm biến** đo đồng thời. Cho hai mảng `sensor1`, `sensor2` (cùng độ dài), trong đó `sensor1[i]`, `sensor2[i]` là điểm dữ liệu thứ `i` của mỗi cảm biến.

Loại cảm biến này có thể bị lỗi: **đúng một** điểm dữ liệu bị **mất**. Sau khi mất, mọi điểm bên phải **dịch sang trái một ô**, và ô **cuối cùng** bị thay bằng một **giá trị ngẫu nhiên**. Đề đảm bảo giá trị ngẫu nhiên này **khác** giá trị bị mất.

Ví dụ: dữ liệu đúng `[1,2,3,4,5]`, mất số `3` → cảm biến có thể trả `[1,2,4,5,7]` (ô cuối có thể là bất kỳ giá trị nào khác `3`).

Biết rằng **nhiều nhất một** cảm biến bị lỗi. Trả về số hiệu cảm biến lỗi (`1` hoặc `2`). Nếu không cảm biến nào lỗi, hoặc **không thể xác định** cảm biến nào lỗi, trả `-1`.

| `sensor1` | `sensor2` | Kết quả | Giải thích |
|---|---|:---:|---|
| `[2,3,4,5]` | `[2,1,3,4]` | `1` | sensor2 đúng; sensor1 mất số `1` (vị trí 1), ô cuối thành `5` |
| `[2,2,2,2,2]` | `[2,2,2,2,5]` | `-1` | không xác định được |
| `[2,3,2,2,3,2]` | `[2,3,2,3,2,7]` | `2` | sensor1 đúng; sensor2 mất số `2` (vị trí 3), ô cuối thành `7` |

**Ràng buộc (theo hiểu biết về đề):** `sensor1.length == sensor2.length`, `1 <= sensor1.length <= 100`, `1 <= sensor1[i], sensor2[i] <= 100`.

In [1]:
from typing import List

examples = [
    ([2, 3, 4, 5], [2, 1, 3, 4], 1),
    ([2, 2, 2, 2, 2], [2, 2, 2, 2, 5], -1),
    ([2, 3, 2, 2, 3, 2], [2, 3, 2, 3, 2, 7], 2),
]
for s1, s2, expected in examples:
    print("sensor1 =", s1, "| sensor2 =", s2, "-> kỳ vọng", expected)

sensor1 = [2, 3, 4, 5] | sensor2 = [2, 1, 3, 4] -> kỳ vọng 1
sensor1 = [2, 2, 2, 2, 2] | sensor2 = [2, 2, 2, 2, 5] -> kỳ vọng -1
sensor1 = [2, 3, 2, 2, 3, 2] | sensor2 = [2, 3, 2, 3, 2, 7] -> kỳ vọng 2


## 2. Nền tảng Python và mô phỏng lỗi

### 2.1. Mô phỏng một cảm biến bị lỗi bằng slicing

Nếu dữ liệu đúng là `correct` và mất phần tử ở vị trí `i`, dữ liệu lỗi là:

```python
correct[:i] + correct[i+1:] + [r]      # r != correct[i]
```

- `correct[:i]`: phần trước vị trí `i` (giữ nguyên).
- `correct[i+1:]`: phần sau vị trí `i` (đã "dịch trái" vì bỏ mất `correct[i]`).
- `[r]`: ô cuối ngẫu nhiên. Phép `+` trên list là **nối list**.

In [2]:
def make_faulty(correct: List[int], i: int, r: int) -> List[int]:
    assert r != correct[i], "giá trị ngẫu nhiên phải khác giá trị bị mất"
    return correct[:i] + correct[i + 1:] + [r]

correct = [1, 2, 3, 4, 5]
print("correct[:2] =", correct[:2], "| correct[3:] =", correct[3:])
print("mất vị trí 2, ô cuối = 7 ->", make_faulty(correct, 2, 7))
print("mất vị trí 4 (cuối), ô cuối = 9 ->", make_faulty(correct, 4, 9))

correct[:2] = [1, 2] | correct[3:] = [4, 5]
mất vị trí 2, ô cuối = 7 -> [1, 2, 4, 5, 7]
mất vị trí 4 (cuối), ô cuối = 9 -> [1, 2, 3, 4, 9]


Nhận xét quan trọng: mất phần tử **cuối cùng** (`i = n - 1`) chỉ làm đổi đúng ô cuối — trông giống hệt việc "ô cuối bị đo sai". Đây là nguồn gốc của trường hợp **không xác định được**.

### 2.2. So sánh hai list bằng `==`

`a == b` so sánh **từng phần tử theo thứ tự** và cả độ dài. Hai list rỗng `[] == []` là `True`.

In [3]:
print([3, 4] == [3, 4], [3, 4] == [4, 3], [] == [])
a = [2, 3, 4, 5]
b = [2, 1, 3, 4]
print("a[1:3] =", a[1:3], "| b[2:4] =", b[2:4], "| bằng nhau?", a[1:3] == b[2:4])

True False True
a[1:3] = [3, 4] | b[2:4] = [3, 4] | bằng nhau? True


### 2.3. Slicing với chỉ số âm `[:-1]`

`a[i:-1]` lấy từ `i` tới **trước phần tử cuối**. Ta sẽ cần "bỏ ô cuối" vì ô cuối của cảm biến lỗi là rác.

In [4]:
a = [2, 3, 4, 5]
print("a[:-1] =", a[:-1], "| a[1:-1] =", a[1:-1], "| a[3:-1] =", a[3:-1], "(rỗng)")

a[:-1] = [2, 3, 4] | a[1:-1] = [3, 4] | a[3:-1] = [] (rỗng)


### 2.4. `for ... else` trong Python

Khối `else` của vòng `for` chỉ chạy khi vòng lặp **kết thúc tự nhiên** (không bị `break`). Tiện để xử lý "không tìm thấy".

In [5]:
for x in [1, 2, 3]:
    if x == 5:
        print("tìm thấy 5")
        break
else:
    print("vòng lặp chạy hết, không gặp break -> không có 5")

vòng lặp chạy hết, không gặp break -> không có 5


## 3. Cách trực giác — brute force mô phỏng đúng đề

Câu hỏi "sensor1 **có thể** là cảm biến lỗi không?" ⇔ tồn tại vị trí `i` sao cho `sensor1` trùng với "`sensor2` bị mất phần tử `i`":

- `sensor1[:n-1] == sensor2[:i] + sensor2[i+1:]` (phần không phải rác khớp), và
- `sensor1[n-1] != sensor2[i]` (ô rác khác giá trị bị mất).

Làm tương tự cho sensor2. Sau đó:

- Chỉ sensor1 có thể lỗi → `1`; chỉ sensor2 → `2`.
- Cả hai đều có thể, hoặc không cái nào (hai mảng giống nhau = không lỗi) → `-1`.

In [6]:
def can_be_faulty(bad: List[int], good: List[int]) -> bool:
    n = len(good)
    for i in range(n):
        if bad[:n - 1] == good[:i] + good[i + 1:] and bad[n - 1] != good[i]:
            return True
    return False

def bad_sensor_brute(sensor1: List[int], sensor2: List[int]) -> int:
    p1 = can_be_faulty(sensor1, sensor2)
    p2 = can_be_faulty(sensor2, sensor1)
    if p1 and not p2:
        return 1
    if p2 and not p1:
        return 2
    return -1

for s1, s2, expected in examples:
    print(bad_sensor_brute(s1, s2), "| kỳ vọng", expected)

1 | kỳ vọng 1
-1 | kỳ vọng -1
2 | kỳ vọng 2


### 3.1. Chi phí brute force

- Thử `n` vị trí; mỗi lần tạo list mới và so sánh `O(n)` → `O(n²)` cho mỗi cảm biến.
- Với `n <= 100` thì chạy tốt, nhưng ta đang làm lại cùng một phép so sánh phần đầu (`good[:i]`) rất nhiều lần.

Câu hỏi: *vị trí bị mất `i` có thể là bất kỳ đâu không, hay bị "ép" vào một chỗ cụ thể?*

## 4. Nút thắt và insight

Gọi `i` là **vị trí đầu tiên** mà `sensor1[i] != sensor2[i]`.

- Nếu không có `i` (hai mảng giống hệt) → không lỗi → `-1`.
- Trước vị trí bị mất `p`, dữ liệu lỗi **trùng** dữ liệu đúng, nên `p <= i`. Có thể `p < i` khi mất một phần tử nằm trong một đoạn giá trị bằng nhau (ví dụ mất một số `2` trong `2,2,2` — kết quả trông y như mất số `2` cuối của đoạn).
- Dù `p` ở đâu, sau khi dịch trái, mảng lỗi trùng mảng đúng trên `[0, i)` và **lệch một ô** từ `i` trở đi:

> **Insight:** Cảm biến `X` có thể lỗi (mảng đúng là `Y`) ⇔ `X[i : n-1] == Y[i+1 : n]`, với `i` là vị trí khác nhau đầu tiên.

Tức: từ vị trí `i`, phần "không rác" của `X` phải bằng `Y` dịch trái một ô.

Kiểm tra hai điều kiện:

- `sensor1` lỗi có thể: `sensor1[i:n-1] == sensor2[i+1:n]`
- `sensor2` lỗi có thể: `sensor2[i:n-1] == sensor1[i+1:n]`

Nếu đúng **một** điều kiện → trả cảm biến tương ứng. Nếu **cả hai** → `-1`.

In [7]:
s1, s2 = [2, 3, 4, 5], [2, 1, 3, 4]
n = len(s1)
i = next(k for k in range(n) if s1[k] != s2[k])
print("vị trí khác đầu tiên i =", i)
print("sensor1 lỗi?  s1[i:n-1] =", s1[i:n - 1], "vs s2[i+1:] =", s2[i + 1:], "->", s1[i:n - 1] == s2[i + 1:])
print("sensor2 lỗi?  s2[i:n-1] =", s2[i:n - 1], "vs s1[i+1:] =", s1[i + 1:], "->", s2[i:n - 1] == s1[i + 1:])

vị trí khác đầu tiên i = 1
sensor1 lỗi?  s1[i:n-1] = [3, 4] vs s2[i+1:] = [3, 4] -> True
sensor2 lỗi?  s2[i:n-1] = [1, 3] vs s1[i+1:] = [4, 5] -> False


`next(generator)` lấy **phần tử đầu tiên** của một generator — cách gọn để "tìm chỉ số đầu tiên thỏa điều kiện" (nếu không có sẽ báo `StopIteration`, nên chỉ dùng khi chắc chắn có).

### 4.1. Còn điều kiện "giá trị ngẫu nhiên khác giá trị bị mất"?

Brute force có kiểm tra `bad[n-1] != good[i]`. Lời giải nhanh không kiểm tra — có sao không? Với **input hợp lệ** (đề đảm bảo đúng như mô tả), có thể chứng minh:

- Nếu cả hai điều kiện dịch đều đúng thì từ `i` trở đi hai mảng xen kẽ dạng `a b a b ...` / `b a b a ...`; khi đó điều kiện "ô rác khác giá trị mất" đúng cho **cả hai** hoặc sai cho **cả hai** (phụ thuộc độ dài đoạn chẵn/lẻ). Sai cả hai nghĩa là input không hợp lệ, nên không xảy ra.
- Nếu chỉ một điều kiện dịch đúng, thì cảm biến đó chắc chắn là cảm biến lỗi thật (cảm biến kia không thể lỗi), và đề đảm bảo có lỗi → điều kiện ô rác tự đúng.

Vì vậy bỏ kiểm tra ô rác không làm thay đổi đáp án trên input hợp lệ. Ta sẽ **kiểm chứng** điều này bằng test ngẫu nhiên ở mục 11.

## 5. Lời giải chính — vị trí khác đầu tiên + so sánh lệch một ô

**Bước 1.** Tìm `i` nhỏ nhất với `sensor1[i] != sensor2[i]`; nếu không có → `-1`.

**Bước 2.** `bad1 = sensor1[i:n-1] == sensor2[i+1:]`, `bad2 = sensor2[i:n-1] == sensor1[i+1:]`.

**Bước 3.** `bad1` và không `bad2` → `1`; `bad2` và không `bad1` → `2`; còn lại → `-1`.

In [8]:
# Bước 1 riêng: tìm vị trí khác đầu tiên bằng for ... else
def first_diff(a: List[int], b: List[int]) -> int:
    for k in range(len(a)):
        if a[k] != b[k]:
            return k
    return -1

print(first_diff([2, 3, 4, 5], [2, 1, 3, 4]), first_diff([2, 2, 2, 2, 2], [2, 2, 2, 2, 5]), first_diff([1, 2], [1, 2]))

1 4 -1


In [9]:
def bad_sensor(sensor1: List[int], sensor2: List[int]) -> int:
    n = len(sensor1)
    i = first_diff(sensor1, sensor2)
    if i == -1:
        return -1
    bad1 = sensor1[i:n - 1] == sensor2[i + 1:]
    bad2 = sensor2[i:n - 1] == sensor1[i + 1:]
    if bad1 and not bad2:
        return 1
    if bad2 and not bad1:
        return 2
    return -1

for s1, s2, expected in examples:
    print(bad_sensor(s1, s2), "| kỳ vọng", expected)

1 | kỳ vọng 1
-1 | kỳ vọng -1
2 | kỳ vọng 2


## 6. TRACE từng bước

**Ví dụ 3:** `sensor1 = [2,3,2,2,3,2]`, `sensor2 = [2,3,2,3,2,7]`, `n = 6`.

| Bước | Nội dung | Giá trị |
|---|---|---|
| 1 | so `k = 0,1,2`: giống nhau; `k = 3`: `2 != 3` | `i = 3` |
| 2a | `sensor1[3:5]` vs `sensor2[4:6]` | `[2,3]` vs `[2,7]` → **False** |
| 2b | `sensor2[3:5]` vs `sensor1[4:6]` | `[3,2]` vs `[3,2]` → **True** |
| 3 | chỉ `bad2` đúng | return `2` |

**Ví dụ 2:** `i = 4` (ô cuối). Cả hai đoạn so sánh đều **rỗng** → `bad1 = bad2 = True` → `-1`.

In [10]:
def trace_bad_sensor(sensor1: List[int], sensor2: List[int]) -> int:
    n = len(sensor1)
    print(f"sensor1={sensor1}\nsensor2={sensor2}")
    i = -1
    for k in range(n):
        same = sensor1[k] == sensor2[k]
        print(f"  k={k}: {sensor1[k]} vs {sensor2[k]} -> {'giống' if same else 'KHÁC'}")
        if not same:
            i = k
            break
    if i == -1:
        print("-> hai mảng giống hệt: không có lỗi, return -1")
        return -1
    a, b = sensor1[i:n - 1], sensor2[i + 1:]
    c2, d = sensor2[i:n - 1], sensor1[i + 1:]
    bad1, bad2 = a == b, c2 == d
    print(f"  i={i}: sensor1[i:n-1]={a} vs sensor2[i+1:]={b} -> sensor1 có thể lỗi: {bad1}")
    print(f"  i={i}: sensor2[i:n-1]={c2} vs sensor1[i+1:]={d} -> sensor2 có thể lỗi: {bad2}")
    ans = 1 if bad1 and not bad2 else 2 if bad2 and not bad1 else -1
    print("-> return", ans)
    return ans

trace_bad_sensor([2, 3, 2, 2, 3, 2], [2, 3, 2, 3, 2, 7])
print()
trace_bad_sensor([2, 2, 2, 2, 2], [2, 2, 2, 2, 5])

sensor1=[2, 3, 2, 2, 3, 2]
sensor2=[2, 3, 2, 3, 2, 7]
  k=0: 2 vs 2 -> giống
  k=1: 3 vs 3 -> giống
  k=2: 2 vs 2 -> giống
  k=3: 2 vs 3 -> KHÁC
  i=3: sensor1[i:n-1]=[2, 3] vs sensor2[i+1:]=[2, 7] -> sensor1 có thể lỗi: False
  i=3: sensor2[i:n-1]=[3, 2] vs sensor1[i+1:]=[3, 2] -> sensor2 có thể lỗi: True
-> return 2

sensor1=[2, 2, 2, 2, 2]
sensor2=[2, 2, 2, 2, 5]
  k=0: 2 vs 2 -> giống
  k=1: 2 vs 2 -> giống
  k=2: 2 vs 2 -> giống
  k=3: 2 vs 2 -> giống
  k=4: 2 vs 5 -> KHÁC
  i=4: sensor1[i:n-1]=[] vs sensor2[i+1:]=[] -> sensor1 có thể lỗi: True
  i=4: sensor2[i:n-1]=[] vs sensor1[i+1:]=[] -> sensor2 có thể lỗi: True
-> return -1


-1

`1 if A else 2 if B else -1` là biểu thức điều kiện **lồng nhau**: đọc từ trái sang — nếu `A` thì 1, ngược lại nếu `B` thì 2, ngược lại -1.

### 6.1. Trace ví dụ 1 và trường hợp hai mảng giống nhau

In [11]:
trace_bad_sensor([2, 3, 4, 5], [2, 1, 3, 4])
print()
trace_bad_sensor([1, 2, 3], [1, 2, 3])

sensor1=[2, 3, 4, 5]
sensor2=[2, 1, 3, 4]
  k=0: 2 vs 2 -> giống
  k=1: 3 vs 1 -> KHÁC
  i=1: sensor1[i:n-1]=[3, 4] vs sensor2[i+1:]=[3, 4] -> sensor1 có thể lỗi: True
  i=1: sensor2[i:n-1]=[1, 3] vs sensor1[i+1:]=[4, 5] -> sensor2 có thể lỗi: False
-> return 1

sensor1=[1, 2, 3]
sensor2=[1, 2, 3]
  k=0: 1 vs 1 -> giống
  k=1: 2 vs 2 -> giống
  k=2: 3 vs 3 -> giống
-> hai mảng giống hệt: không có lỗi, return -1


-1

## 7. Các lời giải khác

### Cách 3 — Hai con trỏ, không tạo bản sao

Slicing tạo list mới (`O(n)` bộ nhớ). Ta có thể so trực tiếp: với `k` từ `i` tới `n - 2`, kiểm tra `sensor1[k] == sensor2[k+1]` (cho bad1) và `sensor2[k] == sensor1[k+1]` (cho bad2), dừng sớm khi cả hai đã sai.

In [12]:
def bad_sensor_pointers(sensor1: List[int], sensor2: List[int]) -> int:
    n = len(sensor1)
    i = 0
    while i < n and sensor1[i] == sensor2[i]:
        i += 1
    if i == n:
        return -1
    bad1 = bad2 = True
    for k in range(i, n - 1):
        if sensor1[k] != sensor2[k + 1]:
            bad1 = False
        if sensor2[k] != sensor1[k + 1]:
            bad2 = False
        if not bad1 and not bad2:
            break
    if bad1 == bad2:
        return -1
    return 1 if bad1 else 2

for s1, s2, expected in examples:
    print(bad_sensor_pointers(s1, s2), "| kỳ vọng", expected)

1 | kỳ vọng 1
-1 | kỳ vọng -1
2 | kỳ vọng 2


Chú ý `while i < n and sensor1[i] == sensor2[i]`: kiểm tra `i < n` **trước** để không vượt chỉ số (short-circuit của `and`).

`bad1 == bad2` gom hai trường hợp "cả hai đúng" và "cả hai sai" → đều trả `-1`.

### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force mô phỏng | thử mọi vị trí bị mất, có kiểm tra ô rác | `O(n²)` | `O(n)` | sát đề nhất, dùng làm "oracle" khi test |
| **2. Vị trí khác đầu tiên + slicing (chính)** | so 2 đoạn lệch 1 ô | `O(n)` | `O(n)` (slice) | rõ ràng, ngắn |
| 3. Hai con trỏ | như cách 2 nhưng so tại chỗ | `O(n)` | `O(1)` | tối ưu bộ nhớ, dừng sớm |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def badSensor(self, sensor1: List[int], sensor2: List[int]) -> int:
        n = len(sensor1)
        i = 0
        while i < n and sensor1[i] == sensor2[i]:
            i += 1
        if i == n:
            return -1
        bad1 = sensor1[i:n - 1] == sensor2[i + 1:]
        bad2 = sensor2[i:n - 1] == sensor1[i + 1:]
        if bad1 and not bad2:
            return 1
        if bad2 and not bad1:
            return 2
        return -1

sol = Solution()
for s1, s2, expected in examples:
    print(sol.badSensor(s1, s2), "| kỳ vọng", expected)

1 | kỳ vọng 1
-1 | kỳ vọng -1
2 | kỳ vọng 2


## 9. Vì sao thuật toán đúng?

Giả sử `Y` đúng, `X` lỗi do mất phần tử ở vị trí `p` và ô cuối là `r != Y[p]`. Khi đó:

- `X[k] = Y[k]` với `k < p`,
- `X[k] = Y[k+1]` với `p <= k <= n-2`,
- `X[n-1] = r`.

Gọi `i` là vị trí khác đầu tiên của `X` và `Y`. Ta có `i >= p` (trước `p` hai mảng giống nhau). Trong đoạn `[p, i)`, `X[k] = Y[k] = Y[k+1]` → `Y[p] = Y[p+1] = ... = Y[i]` — tức mất ở `p` cũng giống mất ở `i`. Từ `i` tới `n-2`, `X[k] = Y[k+1]` → **điều kiện `X[i:n-1] == Y[i+1:n]` đúng**. Vậy cảm biến lỗi thật luôn thỏa điều kiện kiểm tra.

- Nếu cảm biến kia **không** thỏa điều kiện tương ứng thì nó không thể là cảm biến lỗi → đáp án xác định duy nhất → code trả đúng.
- Nếu cả hai đều thỏa, tồn tại hai "kịch bản" phù hợp dữ liệu (mục 4.1 đã chỉ ra cả hai cùng hợp lệ) → không xác định được → `-1` đúng.
- Nếu không có `i` (giống hệt): đề nói "nếu không có lỗi … trả -1" → đúng.

## 10. Độ phức tạp

`n = len(sensor1)`.

- **Time:** `O(n)` — tìm `i` một lượt, so sánh hai đoạn mỗi đoạn `O(n)`.
- **Space:** `O(n)` với slicing; `O(1)` với cách hai con trỏ.

## 11. Bộ test

Vì không có đề gốc trong tay (Premium), ta test kỹ bằng cách **tự sinh dữ liệu hợp lệ đúng như mô tả** và đối chiếu với brute force mô phỏng (có kiểm tra cả điều kiện ô rác).

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.badSensor([2, 3, 4, 5], [2, 1, 3, 4]), 1)
check("đề 2", sol.badSensor([2, 2, 2, 2, 2], [2, 2, 2, 2, 5]), -1)
check("đề 3", sol.badSensor([2, 3, 2, 2, 3, 2], [2, 3, 2, 3, 2, 7]), 2)
check("giống hệt", sol.badSensor([1, 2, 3], [1, 2, 3]), -1)
check("n = 1 khác nhau", sol.badSensor([5], [7]), -1)
check("chỉ khác ô cuối", sol.badSensor([1, 2, 3], [1, 2, 9]), -1)
check("mất phần tử đầu ở sensor2", sol.badSensor([4, 5, 6, 7], [5, 6, 7, 1]), 2)
check("mất phần tử đầu ở sensor1", sol.badSensor([5, 6, 7, 1], [4, 5, 6, 7]), 1)

PASS đề 1: 1
PASS đề 2: -1
PASS đề 3: 2
PASS giống hệt: -1
PASS n = 1 khác nhau: -1
PASS chỉ khác ô cuối: -1
PASS mất phần tử đầu ở sensor2: 2
PASS mất phần tử đầu ở sensor1: 1


In [15]:
import random
random.seed(1826)

def random_case(n, vmax):
    correct = [random.randint(1, vmax) for _ in range(n)]
    mode = random.choice(["none", "s1", "s2"])
    if mode == "none":
        return correct[:], correct[:]
    p = random.randrange(n)
    r = random.choice([v for v in range(1, vmax + 2) if v != correct[p]])
    faulty = make_faulty(correct, p, r)
    return (faulty, correct) if mode == "s1" else (correct, faulty)

solvers = [bad_sensor, bad_sensor_pointers, Solution().badSensor]
stats = {1: 0, 2: 0, -1: 0}
for _ in range(5000):
    s1, s2 = random_case(random.randint(1, 8), random.choice([2, 3, 10]))
    expected = bad_sensor_brute(s1, s2)
    for f in solvers:
        assert f(s1, s2) == expected, (f.__name__, s1, s2, expected)
    stats[expected] += 1
print("Mọi cách khớp brute force trên 5000 input hợp lệ ngẫu nhiên (seed 1826)")
print("Phân bố đáp án:", stats)

Mọi cách khớp brute force trên 5000 input hợp lệ ngẫu nhiên (seed 1826)
Phân bố đáp án: {1: 882, 2: 898, -1: 3220}


Giá trị nhỏ (`vmax = 2, 3`) tạo nhiều đoạn bằng nhau và xen kẽ `a b a b` — đúng những chỗ dễ sinh trường hợp "không xác định". Phân bố đáp án cho thấy cả ba loại kết quả đều được kiểm tra.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Trả về cảm biến ngay khi một điều kiện đúng, không xét điều kiện kia

In [16]:
def wrong_first_match(sensor1, sensor2):
    i = first_diff(sensor1, sensor2)
    if i == -1:
        return -1
    n = len(sensor1)
    if sensor1[i:n - 1] == sensor2[i + 1:]:
        return 1
    if sensor2[i:n - 1] == sensor1[i + 1:]:
        return 2
    return -1

print("sai:", wrong_first_match([2, 2, 2, 2, 2], [2, 2, 2, 2, 5]), "| đúng:", bad_sensor([2, 2, 2, 2, 2], [2, 2, 2, 2, 5]))

sai: 1 | đúng: -1


### Lỗi 2: So sánh cả ô cuối (không bỏ rác)

Viết `sensor1[i:] == sensor2[i+1:]` → độ dài lệch nhau một → luôn `False`.

In [17]:
s1, s2 = [2, 3, 4, 5], [2, 1, 3, 4]
i = 1
print("sai: s1[i:] =", s1[i:], "vs s2[i+1:] =", s2[i + 1:], "->", s1[i:] == s2[i + 1:])
print("đúng: s1[i:-1] =", s1[i:-1], "vs s2[i+1:] =", s2[i + 1:], "->", s1[i:-1] == s2[i + 1:])

sai: s1[i:] = [3, 4, 5] vs s2[i+1:] = [3, 4] -> False
đúng: s1[i:-1] = [3, 4] vs s2[i+1:] = [3, 4] -> True


### Lỗi 3: Nhầm chiều dịch

Kiểm tra `sensor1[i+1:] == sensor2[i:n-1]` để kết luận **sensor1** lỗi là **ngược**: điều kiện đó nói `sensor2` bị dịch trái so với `sensor1`, tức **sensor2** lỗi. Mẹo nhớ: *mảng lỗi "đi trước" một ô — `bad[k] == good[k+1]`*.

### Lỗi 4: Quên trường hợp hai mảng giống hệt

Nếu bỏ dòng `if i == n: return -1`, cả hai cách vẫn ra `-1` vì hai đoạn so sánh đều rỗng (`bad1 = bad2 = True`) — nhưng đó là **đúng nhờ may mắn**, người đọc code không thấy được ý đồ. Còn với cách dùng `first_diff` trả `-1`, quên kiểm tra sẽ khiến `sensor1[-1:n-1]` bị hiểu sai hoàn toàn. Hãy xử lý tường minh.

### Lỗi 5: Đọc `sensor1[i]` khi `i == n` trong vòng `while`

`while sensor1[i] == sensor2[i] and i < n` → `IndexError` khi hai mảng giống nhau. Phải đặt `i < n` **trước**.

In [18]:
def wrong_order_condition(a, b):
    i = 0
    while a[i] == b[i] and i < len(a):
        i += 1
    return i

try:
    wrong_order_condition([1, 2], [1, 2])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 13. Mẹo nhận dạng pattern

- "Hai dãy gần giống nhau, một dãy bị **chèn/xóa một phần tử**" → tìm **vị trí khác đầu tiên**, rồi so phần còn lại với **độ lệch 1**.
- Đây là cùng ý tưởng với bài **One Edit Distance** (161) và kiểm tra "xóa một ký tự thành palindrome" (680).
- "Không xác định được" → kiểm tra **mọi kịch bản** phù hợp dữ liệu; nếu hơn một → trả giá trị đặc biệt.
- Đề mô phỏng phức tạp → viết **brute force mô phỏng** trước, dùng nó làm oracle để test lời giải nhanh.

## 14. Biến thể

### 14.1. One Edit Distance (LeetCode 161, Premium) — hai chuỗi cách nhau đúng một phép chèn/xóa/thay

Dùng lại "vị trí khác đầu tiên + so phần còn lại lệch 0 hoặc 1".

In [19]:
def is_one_edit_distance(s: str, t: str) -> bool:
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1:
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            if len(s) == len(t):
                return s[i + 1:] == t[i + 1:]      # thay một ký tự
            return s[i:] == t[i + 1:]              # chèn vào s (xóa khỏi t)
    return len(t) - len(s) == 1                    # khác ở ký tự cuối của t

print(is_one_edit_distance("ab", "acb"), is_one_edit_distance("cab", "ad"), is_one_edit_distance("1203", "1213"))

True False True


### 14.2. Khôi phục dữ liệu đúng khi biết cảm biến lỗi

Nếu biết `sensor1` lỗi (`sensor2` đúng) thì dữ liệu đúng chính là `sensor2`. Thú vị hơn: biết `sensor1` đúng, tìm **vị trí bị mất** trong `sensor2` (vị trí nhỏ nhất có thể) — đó chính là `i` khác nhau đầu tiên.

In [20]:
def dropped_position(correct: List[int], faulty: List[int]) -> int:
    return first_diff(correct, faulty)

print(dropped_position([2, 3, 2, 2, 3, 2], [2, 3, 2, 3, 2, 7]))   # 3

3


## 15. Bài tập luyện tập

**Bài 1 (tính tay):** Dữ liệu đúng `[5, 1, 4, 4, 2]`, mất phần tử ở vị trí 1, ô cuối thay bằng `9`. Mảng lỗi là gì?

**Bài 2 (tính tay):** `sensor1 = [1, 2, 1, 2]`, `sensor2 = [2, 1, 2, 9]`. Tìm `i`, tính `bad1`, `bad2` và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện hàm kiểm tra "`bad` có thể là bản lỗi của `good`" theo insight (không kiểm tra ô rác).

In [21]:
def ex3_shift_match(bad: List[int], good: List[int]) -> bool:
    # TODO 1: tìm i đầu tiên bad[i] != good[i]; không có -> return False
    # TODO 2: return bad[i:n-1] == good[i+1:]
    pass

# assert ex3_shift_match([2, 3, 4, 5], [2, 1, 3, 4]) == True
# assert ex3_shift_match([2, 1, 3, 4], [2, 3, 4, 5]) == False

**Bài 4 (tự viết):** Viết lại `badSensor` theo cách **hai con trỏ** mà không nhìn mục 7.

In [22]:
def ex4_bad_sensor(sensor1: List[int], sensor2: List[int]) -> int:
    # TODO
    pass

# for s1, s2, expected in examples:
#     assert ex4_bad_sensor(s1, s2) == expected

**Bài 5 (biến thể):** Viết `ex5_one_deletion(a, b)` — `len(b) == len(a) - 1`; trả `True` nếu xóa **đúng một** phần tử của `a` thì được `b`.

In [23]:
def ex5_one_deletion(a: List[int], b: List[int]) -> bool:
    # TODO
    pass

# assert ex5_one_deletion([1, 2, 3], [1, 3]) == True
# assert ex5_one_deletion([1, 2, 3], [3, 1]) == False
# assert ex5_one_deletion([7], []) == True

**Bài 6 (mini challenge):** Ba cảm biến `s1, s2, s3`, nhiều nhất một cái lỗi (cùng kiểu lỗi). Viết `ex6_bad_of_three(s1, s2, s3)` trả số hiệu cảm biến lỗi, hoặc `-1` nếu không lỗi/không xác định. Gợi ý: nếu hai cảm biến giống hệt nhau thì chúng đều đúng.

In [24]:
def ex6_bad_of_three(s1: List[int], s2: List[int], s3: List[int]) -> int:
    # TODO
    pass

# assert ex6_bad_of_three([1, 2, 3], [1, 2, 3], [1, 3, 9]) == 3
# assert ex6_bad_of_three([1, 2, 3], [1, 2, 3], [1, 2, 3]) == -1
# assert ex6_bad_of_three([2, 3, 9], [1, 2, 3], [1, 2, 3]) == 1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | bỏ `correct[1]` | dịch trái phần sau | `[5, 4, 4, 2, 9]` |
| 2 | `i = 0` | `bad1`: `[1,2,1]` vs `[1,2,9]` | `bad2`: `[2,1,2]` vs `[2,1,2]` → `2` |
| 3 | `first_diff` | `n = len(good)` | nhớ trường hợp không có `i` |
| 4 | `while` tìm `i` | duyệt `k` từ `i` tới `n-2` | `bad1 == bad2` → `-1` |
| 5 | tìm vị trí khác đầu tiên giữa `a` và `b` | không có → xóa phần tử cuối | `a[:i] + a[i+1:] == b` |
| 6 | cặp giống hệt = đúng | cảm biến còn lại: giống → `-1`, khác → nó lỗi | không có cặp giống → dùng `bad_sensor` trên từng cặp (nâng cao) |

## 17. Đáp án tham khảo

**Bài 1:** `[5, 4, 4, 2, 9]`.

**Bài 2:** `i = 0`; `bad1`: `sensor1[0:3] = [1,2,1]` vs `sensor2[1:] = [1,2,9]` → False; `bad2`: `sensor2[0:3] = [2,1,2]` vs `sensor1[1:] = [2,1,2]` → True → đáp án **2**.

In [25]:
print(make_faulty([5, 1, 4, 4, 2], 1, 9))
assert trace_bad_sensor([1, 2, 1, 2], [2, 1, 2, 9]) == 2

def ex3_shift_match_ans(bad, good):
    n = len(good)
    i = first_diff(bad, good)
    if i == -1:
        return False
    return bad[i:n - 1] == good[i + 1:]

def ex4_bad_sensor_ans(sensor1, sensor2):
    return bad_sensor_pointers(sensor1, sensor2)

def ex5_one_deletion_ans(a, b):
    if len(b) != len(a) - 1:
        return False
    for i in range(len(b)):
        if a[i] != b[i]:
            return a[i + 1:] == b[i:]
    return True

def ex6_bad_of_three_ans(s1, s2, s3):
    sensors = [s1, s2, s3]
    for x in range(3):
        for y in range(x + 1, 3):
            if sensors[x] == sensors[y]:
                z = 3 - x - y                  # chỉ số còn lại
                return -1 if sensors[z] == sensors[x] else z + 1
    # không có cặp nào giống hệt: không thể xảy ra nếu nhiều nhất 1 cảm biến lỗi
    return -1

assert ex3_shift_match_ans([2, 3, 4, 5], [2, 1, 3, 4]) == True
assert ex3_shift_match_ans([2, 1, 3, 4], [2, 3, 4, 5]) == False
for s1, s2, expected in examples:
    assert ex4_bad_sensor_ans(s1, s2) == expected
assert ex5_one_deletion_ans([1, 2, 3], [1, 3]) == True
assert ex5_one_deletion_ans([1, 2, 3], [3, 1]) == False
assert ex5_one_deletion_ans([7], []) == True
assert ex6_bad_of_three_ans([1, 2, 3], [1, 2, 3], [1, 3, 9]) == 3
assert ex6_bad_of_three_ans([1, 2, 3], [1, 2, 3], [1, 2, 3]) == -1
assert ex6_bad_of_three_ans([2, 3, 9], [1, 2, 3], [1, 2, 3]) == 1
print("Đáp án tham khảo PASS toàn bộ test!")

[5, 4, 4, 2, 9]
sensor1=[1, 2, 1, 2]
sensor2=[2, 1, 2, 9]
  k=0: 1 vs 2 -> KHÁC
  i=0: sensor1[i:n-1]=[1, 2, 1] vs sensor2[i+1:]=[1, 2, 9] -> sensor1 có thể lỗi: False
  i=0: sensor2[i:n-1]=[2, 1, 2] vs sensor1[i+1:]=[2, 1, 2] -> sensor2 có thể lỗi: True
-> return 2
Đáp án tham khảo PASS toàn bộ test!


Ghi chú bài 6: với ba cảm biến mà nhiều nhất một cái lỗi, **hai cái đúng luôn giống hệt nhau**, nên luôn tìm được một cặp giống nhau. Cảm biến thứ ba nếu khác thì chắc chắn là cảm biến lỗi — có thêm "nhân chứng" thứ ba giúp luôn xác định được, khác với bài gốc chỉ có hai cảm biến.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def badSensor(self, sensor1: List[int], sensor2: List[int]) -> int:
        n, i = len(sensor1), 0
        while i < n and sensor1[i] == sensor2[i]:   # vị trí khác đầu tiên
            i += 1
        if i == n:
            return -1                               # giống hệt: không lỗi
        bad1 = sensor1[i:n - 1] == sensor2[i + 1:]  # sensor1 "đi trước" một ô
        bad2 = sensor2[i:n - 1] == sensor1[i + 1:]
        if bad1 != bad2:
            return 1 if bad1 else 2
        return -1                                   # cả hai đều có thể -> không xác định
```

**Câu nhớ:** *"Tìm chỗ khác đầu tiên; mảng lỗi đi trước một ô; bỏ ô cuối rác; cả hai cùng khớp thì -1."*

**Checklist:**

- [ ] Xử lý hai mảng giống hệt → `-1`?
- [ ] Bỏ ô cuối (`[i:n-1]`) của mảng bị nghi lỗi khi so sánh?
- [ ] Đúng chiều dịch: `bad[k] == good[k+1]`?
- [ ] Xét **cả hai** khả năng trước khi kết luận?
- [ ] Đặt `i < n` trước khi đọc `sensor1[i]`?
- [ ] (Premium) Đã đối chiếu lại đề nếu có tài khoản?